**Machine Learning Lab**

# 8. Model Enhancement: Feature Engineering, Class Imbalance Handling, and Hyperparameter Optimization

In this notebook, we will explore key techniques to improve machine learning models beyond their initial performance. The focus will be on three major areas: **feature engineering** to create more informative inputs, **class imbalance handling** to ensure fair learning across underrepresented classes, and **hyperparameter tuning** to optimize model performance. By applying these methods, you will gain hands-on experience with practical strategies that make models more accurate, robust, and reliable.

---

The dataset used in this notebook is the [Cirrhosis Patient Survival Prediction dataset](https://archive.ics.uci.edu/dataset/878/cirrhosis+patient+survival+prediction+dataset-1), which originates from a Mayo Clinic study on primary biliary cirrhosis (PBC) conducted between 1974 and 1984. It contains data from 418 patients, described by 17 clinical features that are either real-valued or categorical. The prediction task is to classify a patient's survival state, labeled as death (D), censored (C), or censored due to liver transplantation (CL). Since the data includes sensitive attributes such as age and gender, it provides an opportunity to explore not only technical aspects of modeling but also considerations around fairness and ethics in healthcare applications.

## 1. Preliminaries

Import the core libraries we'll use for data handling, numeric work, plotting, and the train/test split.

In [ ]:
# General Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Preprocessing and Result Display
from sklearn.model_selection import train_test_split

This notebook uses `cirrhosis.csv`. Run the cell below and upload it (found in this notebook's folder in the repo) when prompted.

In [ ]:
from google.colab import files
uploaded = files.upload()  # select cirrhosis.csv

Load the dataset and preview it.

In [ ]:
df = pd.read_csv('cirrhosis.csv')
df.head()

Check the overall size of the dataset.

In [ ]:
df.shape

Define feature and label sets.

In [ ]:
# Feature set
X = df.drop(['Status', 'ID'], axis=1)
X.head()

`ID` is dropped along with `Status` since it's just a row identifier, not a predictive feature.

In [ ]:
# Label set
y = df['Status']
y.head()

We will do a train/test split with an 80-20 ratio.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 2. Feature Engineering

Feature engineering is the process of shaping your raw data so that your machine learning model can understand it better. This could mean turning categories into numbers, scaling values so they're comparable, filling in missing info, or even creating new features from the existing ones. Doing it well can make a big difference in how accurately your model predicts, sometimes even more than which algorithm you pick.

### 2.1 Label Encoding

We need a lot of preprocessing for this dataset. Let's start with label encoding. Label encoding turns categorical columns into numerical ones — it enumerates categorical values and maps them to an integer. Before going into this step, let's look at `X_train`.

List every column in `X_train`.

In [ ]:
X_train.columns

Preview a few rows to see which columns hold text (categorical) values.

In [ ]:
X_train.head(3)

Only the categorical columns need label encoding. By looking at the data, we can see that these columns are `Drug`, `Sex`, `Ascites`, `Hepatomegaly`, `Spiders`, `Edema`.

Before starting label encoding, we will do these things:
1. We will modify a copy of the DataFrame instead of the original, to keep the original values unchanged.
2. There are some NaN (missing) values in the dataset. These values will remain as they are for now — we'll deal with them separately in section 2.2.
3. The same transformation should be applied to the test data. But for the test data we will use `transform()` instead of `fit_transform()`, so the encoding stays consistent between train and test sets.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
import numpy as np

categorical_cols = ['Drug', 'Sex', 'Ascites', 'Hepatomegaly', 'Spiders', 'Edema']

# Create a copy to avoid modifying the original dataframe directly if needed later
X_train_encoded = X_train.copy()
X_test_encoded = X_test.copy()

for col in categorical_cols:
    oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=np.nan)

    # Fit on train and transform both train & test
    X_train_encoded[col] = oe.fit_transform(X_train_encoded[[col]]) + 1
    X_test_encoded[col] = oe.transform(X_test_encoded[[col]]) + 1

display(X_train_encoded.head())

We will also label encode `y_train` and `y_test`. Before labeling, it looks like this:

In [ ]:
y_train.head()

After label encoding:

In [ ]:
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=np.nan)
y_train = oe.fit_transform(y_train.values.reshape(-1, 1))
y_test = oe.transform(y_test.values.reshape(-1, 1))

# Ravel will flatten the values
y_train = y_train.ravel()
y_test = y_test.ravel()

# Label encoding turns the single column into a numpy array. We will retransform them into pandas
# Series. This is usually not necessary, but for some later steps we will do this.
y_train = pd.Series(y_train)
y_test = pd.Series(y_test)

y_train

Double-check the encoded labels look right.

In [ ]:
y_train

### 2.2 Removing Missing Values

Take another look at the encoded training features before handling missing values.

In [ ]:
X_train_encoded.head()

Now let's focus on the values written as `NaN`. They are blank/missing values. Most machine learning models can't work with missing values, so we need to handle them somehow.

Our first strategy will be to drop them altogether.

In [ ]:
X_test_encoded.shape, y_test.shape

Drop every row with at least one missing value, keeping `X` and `y` aligned by index.

In [ ]:
print(f'Before removing missing values - X_train_encoded: {X_train_encoded.shape}')
print(f'Before removing missing values - X_test_encoded: {X_test_encoded.shape}')

X_train_encoded_dropped = X_train_encoded.dropna().reset_index(drop=True)
y_train_dropped = y_train[X_train_encoded_dropped.index].reset_index(drop=True)
X_test_encoded_dropped = X_test_encoded.dropna().reset_index(drop=True)
y_test_dropped = y_test[X_test_encoded_dropped.index].reset_index(drop=True)

print(f'After removing missing values: {X_train_encoded_dropped.shape}')
print(f'After removing missing values: {X_test_encoded_dropped.shape}')

There are other things we can do, but before going there we will first do a performance check on this prepared dataset. We will have to repeat this training/evaluation process many times later, so let's write a reusable function that trains several models and reports their accuracy.

Import the classifiers we'll compare throughout the rest of this notebook.

In [ ]:
# Machine learning models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB

Define `train_and_evaluate_models`, which fits every model on the given training data and prints/stores its accuracy on the test data. `XGBClassifier` is included alongside the other ensemble methods, since we imported it above.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

def train_and_evaluate_models(X_train, y_train, X_test, y_test):

    result = {}

    models = [
        LogisticRegression(max_iter=1000, random_state=42),
        RandomForestClassifier(random_state=42),
        AdaBoostClassifier(random_state=42),
        GradientBoostingClassifier(random_state=42),
        XGBClassifier(random_state=42),
        SVC(random_state=42),
        DecisionTreeClassifier(random_state=42),
        KNeighborsClassifier(), # KNeighborsClassifier does not have random_state
        MLPClassifier(random_state=42),
        GaussianNB() # GaussianNB does not have random_state
    ]

    for model in models:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        print(f"{model.__class__.__name__} Accuracy: {accuracy_score(y_test, y_pred)}")
        result[model.__class__.__name__] = accuracy_score(y_test, y_pred)

    return result

We will train and test on `X_train_encoded_dropped`, `y_train_dropped`, `X_test_encoded_dropped`, `y_test_dropped`.

In [ ]:
# Now call the function with your data
result_encoded_dropped = train_and_evaluate_models(X_train_encoded_dropped, y_train_dropped, X_test_encoded_dropped, y_test_dropped)

TERRIBLE PERFORMANCE! Now, let's try to improve this.

### 2.3 Imputing Missing Values

Instead of removing missing values altogether, we will now impute them. Imputation means filling in the missing values with substitute values. They can be filled using prior knowledge of the dataset, the data scientist's judgment, statistical single imputation, or even more complex multiple imputation.

We will use something simpler. First, let's have a look at which columns have missing values.

In [ ]:
X_train_encoded.isna().sum()

Almost all of them have missing values. As we know, some of these columns were originally categorical and some were numerical — different types of columns need different approaches. First we handle the numerical columns with simple imputation: missing values are filled with the mean of the other values in their column.

In [ ]:
from sklearn.impute import SimpleImputer

# Identify numerical columns
numerical_cols = X_train_encoded.columns.difference(categorical_cols)

# Create copies of the original
X_train_imputed = X_train_encoded.copy()
X_test_imputed = X_test_encoded.copy()

# Create an imputer object with mean strategy
imputer = SimpleImputer(strategy='mean')

# Apply imputation to numerical columns in train and test sets
X_train_imputed[numerical_cols] = imputer.fit_transform(X_train_encoded[numerical_cols])
X_test_imputed[numerical_cols] = imputer.transform(X_test_encoded[numerical_cols])

X_train_imputed.head()

Let's look at a before/after comparison of how many missing values remain in each column.

In [ ]:
missing_before = X_train_encoded.isna().sum()
missing_after = X_train_imputed.isna().sum()

# Combine the missing value counts into a DataFrame
missing_comparison = pd.DataFrame({
    'Before Imputation': missing_before,
    'After Imputation': missing_after
})

# Display the table, filtering to show only columns that had missing values
display(missing_comparison[missing_comparison['Before Imputation'] > 0])

Now we need to deal with the categorical columns. There are several strategies we can follow:

1. **Statistical Imputation**: Fill missing values with a significant value — for example, the mode (the value most frequently occurring in a column).
2. **New Category**: Keep missing values as a new category.
3. **Predictive Imputation**: Apply another machine learning model to fill in the missing values.

We will go with option 2 for now. When doing label encoding earlier, we started indexing from 1, so we can simply set all missing values to 0.

In [ ]:
X_train_imputed.fillna(0, inplace=True)
X_test_imputed.fillna(0, inplace=True)

X_train_imputed.isna().sum()

Let's train and test again, now on the fully imputed data.

In [ ]:
result_imputed = train_and_evaluate_models(X_train_imputed, y_train, X_test_imputed, y_test)

Let's compare the two approaches (dropping vs. imputing) side by side.

In [ ]:
#@title Model Accuracy Comparison

# Tableau 10 (colorblind-friendly and widely used in publications)
colors = [
    "#1f77b4",  # Blue
    "#ff7f0e",  # Orange
    "#2ca02c",  # Green
    "#d62728",  # Red
    "#9467bd",  # Purple
    "#8c564b",  # Brown
    "#e377c2",  # Pink
    "#7f7f7f",  # Gray
    "#bcbd22",  # Olive
    "#17becf"   # Cyan
]

plt.rcParams['axes.prop_cycle'] = plt.cycler(color=colors)

models = list(result_imputed.keys())
accuracies_dropped = [result_encoded_dropped.get(model, 0) for model in models]
accuracies_imputed = list(result_imputed.values())

x = np.arange(len(models))  # the label locations
width = 0.35  # the width of the bars

fig, ax = plt.subplots(figsize=(14, 7))

rects2 = ax.bar(x - width/2, accuracies_dropped, width, label='Dropped Missing Values')
rects1 = ax.bar(x + width/2, accuracies_imputed, width, label='Imputed Missing Values')

# Add some text for labels, title and custom x-axis tick labels, etc.
ax.set_ylabel('Accuracy')
ax.set_title('Model Accuracy Comparison: Imputed vs. Dropped Missing Values')
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=45, ha="right")
ax.legend()

plt.tight_layout()
plt.show()

GREAT IMPROVEMENT! But, can we do better?

### 2.4 Feature Scaling / Normalization

Feature scaling brings all values into a specific range. It's especially useful for models that are distance- or gradient-based (KNN, SVM, MLP). We will apply normalization to the dataset.

Usually normalization is applied only to numerical columns, but we'll also apply it to the (already integer-encoded) categorical columns here, since the values within each category will still be scaled consistently.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Create a MinMaxScaler object
# We could also use StandardScaler
scaler = MinMaxScaler()

# Apply scaling to the imputed data and create new variables
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

# Convert back to DataFrame to keep column names
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train_imputed.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test_imputed.columns)

X_train_scaled.head()

Train and evaluate again on the scaled data.

In [ ]:
result_scaled = train_and_evaluate_models(X_train_scaled, y_train, X_test_scaled, y_test)

Compare all three approaches so far: dropped, imputed, and imputed+scaled.

In [ ]:
#@title Model Accuracy Comparison

models = list(result_scaled.keys())
accuracies_dropped = [result_encoded_dropped.get(model, 0) for model in models]
accuracies_imputed = [result_imputed.get(model, 0) for model in models]
accuracies_imputed_scaled = list(result_scaled.values())

x = np.arange(len(models))  # the label locations
width = 0.25  # the width of the bars

fig, ax = plt.subplots(figsize=(16, 8))

rects1 = ax.bar(x - width, accuracies_dropped, width, label='Dropped Missing Values')
rects2 = ax.bar(x, accuracies_imputed, width, label='Imputed Missing Values')
rects3 = ax.bar(x + width, accuracies_imputed_scaled, width, label='Imputed and Scaled Values')

# Add some text for labels, title and custom x-axis tick labels, etc.
ax.set_ylabel('Accuracy')
ax.set_title('Model Accuracy Comparison: Dropped vs. Imputed vs. Imputed and Scaled Missing Values')
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=45, ha="right")
ax.legend()

plt.tight_layout()
plt.show()

### 2.5 One-Hot Encoding

We used label encoding to enumerate the categorical columns, but the numbers assigned don't carry any real order or meaning — a model might mistakenly treat category `3` as "greater than" category `1`.

A solution to this is one-hot encoding. This turns a categorical column into multiple binary columns, one per category, so no false ordering is implied.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# Identify categorical columns (from earlier definition)
categorical_cols = ['Drug', 'Sex', 'Ascites', 'Hepatomegaly', 'Spiders', 'Edema']

# Create a OneHotEncoder object
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Apply one-hot encoding to the categorical columns in the scaled and imputed data
X_train_ohe = ohe.fit_transform(X_train_scaled[categorical_cols])
X_test_ohe = ohe.transform(X_test_scaled[categorical_cols])

# Create DataFrames from the one-hot encoded arrays with appropriate column names
# Get the feature names for the one-hot encoded columns
ohe_feature_names = ohe.get_feature_names_out(categorical_cols)
X_train_ohe_df = pd.DataFrame(X_train_ohe, columns=ohe_feature_names, index=X_train_scaled.index)
X_test_ohe_df = pd.DataFrame(X_test_ohe, columns=ohe_feature_names, index=X_test_scaled.index)

# Drop the original categorical columns from the scaled and imputed dataframes
X_train_processed = X_train_scaled.drop(columns=categorical_cols)
X_test_processed = X_test_scaled.drop(columns=categorical_cols)

# Concatenate the one-hot encoded dataframes with the remaining numerical columns
X_train_ohe_processed = pd.concat([X_train_processed, X_train_ohe_df], axis=1)
X_test_ohe_processed = pd.concat([X_test_processed, X_test_ohe_df], axis=1)

X_train_ohe_processed.head()

List all the columns in the DataFrame now — notice how each original category column expanded into several binary columns.

In [ ]:
X_train_ohe_processed.columns

Train and evaluate on the one-hot encoded features.

In [ ]:
result_ohe = train_and_evaluate_models(X_train_ohe_processed, y_train, X_test_ohe_processed, y_test)

Compare all approaches so far: dropped, imputed, imputed+scaled, and one-hot encoded.

In [ ]:
#@title Model Accuracy Comparison

models = list(result_ohe.keys()) # Use keys from ohe as it likely has all models
accuracies_dropped = [result_encoded_dropped.get(model, 0) for model in models]
accuracies_imputed = [result_imputed.get(model, 0) for model in models]
accuracies_imputed_scaled = [result_scaled.get(model, 0) for model in models]
accuracies_ohe = list(result_ohe.values())

x = np.arange(len(models))  # the label locations
width = 0.15  # the width of the bars

fig, ax = plt.subplots(figsize=(20, 10))

rects1 = ax.bar(x - width*2, accuracies_dropped, width, label='Dropped Missing Values')
rects2 = ax.bar(x - width, accuracies_imputed, width, label='Imputed Missing Values')
rects3 = ax.bar(x, accuracies_imputed_scaled, width, label='Imputed and Scaled Values')
rects5 = ax.bar(x + width, accuracies_ohe, width, label='One-Hot Encoded, Imputed and Scaled Values')

# Add some text for labels, title and custom x-axis tick labels, etc.
ax.set_ylabel('Accuracy')
ax.set_title('Model Accuracy Comparison: Different Preprocessing Methods')
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=45, ha="right")
ax.legend()

plt.tight_layout()
plt.show()

Check how many total columns we ended up with after one-hot encoding.

In [ ]:
X_train_ohe_processed.shape

### 2.6 Feature Selection

Feature selection is picking out the features that actually matter for your model and getting rid of the noisy or irrelevant ones. This makes your model faster, less likely to overfit, and easier to understand. You can do it in different ways, like checking correlations, using recursive elimination, or letting models like Lasso or Random Forest tell you which features are important.

Here we will use `RFECV`, which is especially useful for smaller datasets. RFECV (Recursive Feature Elimination with Cross-Validation) automatically picks the best features by repeatedly training the model, ranking features by importance, and removing the least important ones step by step. At each stage, it uses cross-validation to check performance, and in the end it selects the number of features that gives the best result — no manual guessing needed.

To rank feature importance, a **logistic regression** model will be used.

In [ ]:
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

log_reg = LogisticRegression(max_iter=1000, random_state=42)

# Initialize RFECV with Logistic Regression and 5-fold cross-validation
# Using StratifiedKFold to maintain class distribution in folds
rfecv = RFECV(estimator=log_reg, step=1, cv=StratifiedKFold(5), scoring='accuracy')

rfecv.fit(X_train_ohe_processed, y_train)

print(f"Optimal number of features : {rfecv.n_features_}")
print("Selected features:")
print(X_train_ohe_processed.columns[rfecv.support_])

Keep only the selected features, then train and evaluate again.

In [ ]:
# Get the selected features
selected_features = X_train_ohe_processed.columns[rfecv.support_]

# Filter the training and testing data to include only the selected features
X_train_selected = X_train_ohe_processed[selected_features]
X_test_selected = X_test_ohe_processed[selected_features]

# Train and evaluate models using the selected features
result_selected_features = train_and_evaluate_models(X_train_selected, y_train, X_test_selected, y_test)

Add this result to the running comparison chart.

In [ ]:
#@title Model Accuracy Comparison

models = list(result_ohe.keys()) # Use keys from ohe as it likely has all models
accuracies_dropped = [result_encoded_dropped.get(model, 0) for model in models]
accuracies_imputed = [result_imputed.get(model, 0) for model in models]
accuracies_imputed_scaled = [result_scaled.get(model, 0) for model in models]
accuracies_ohe = list(result_ohe.values())
accuracies_selected = [result_selected_features.get(model, 0) for model in models]

x = np.arange(len(models))  # the label locations
width = 0.12  # the width of the bars

fig, ax = plt.subplots(figsize=(22, 10))

rects1 = ax.bar(x - width*2.5, accuracies_dropped, width, label='Dropped Missing Values')
rects2 = ax.bar(x - width*1.5, accuracies_imputed, width, label='Imputed Missing Values')
rects3 = ax.bar(x - width*0.5, accuracies_imputed_scaled, width, label='Imputed and Scaled Values')
rects5 = ax.bar(x + width*0.5, accuracies_ohe, width, label='One-Hot Encoded, Imputed and Scaled Values')
rects6 = ax.bar(x + width*1.5, accuracies_selected, width, label='Selected Features (OHE, Imputed, Scaled)')

# Add some text for labels, title and custom x-axis tick labels, etc.
ax.set_ylabel('Accuracy')
ax.set_title('Model Accuracy Comparison: Different Preprocessing Methods and Feature Selection')
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=45, ha="right")
ax.legend()

plt.tight_layout()
plt.show()

## 3. Imbalance Removal

Class imbalance is when the distribution of different classes in the dataset is not equal. Almost every dataset is more or less imbalanced, but sometimes the imbalance is severe enough to hurt model performance.

Let's check the imbalance for this dataset.

In [ ]:
y_train.value_counts()

Visualize the class distribution as a pie chart.

In [ ]:
#@title Pie Chart of Class Distribution
plt.figure(figsize=(8, 8))
y_train.value_counts().plot.pie(autopct='%1.1f%%', startangle=90)
plt.title('Class Distribution in Training Data')
plt.ylabel('') # Remove default y-label
plt.show()

The 3 classes are pretty imbalanced, especially the sample size for class 1 (`CL`). Imbalance is usually addressed by adding new samples to the underrepresented class(es). Oversampling can be done by duplicating random rows, or by generating synthetic data.

We'll go with the second approach. **SMOTE** is a commonly used method for creating synthetic samples of a minority class. We'll apply it to boost class 1 specifically, while leaving the other two classes untouched (their target counts below match their current size, since they're already reasonably represented).

In [ ]:
from imblearn.over_sampling import SMOTE
import pandas as pd

# Define target counts: only oversample class 1.0 up to 80 samples (instead of balancing fully)
target_counts = {0.0: 188, 2.0: 125, 1.0: 80}

# Apply SMOTE with custom sampling_strategy
smote = SMOTE(sampling_strategy=target_counts, random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print("Shape of training data before SMOTE:", X_train_scaled.shape)
print("Shape of training data after SMOTE:", X_train_resampled.shape)

print("\nClass distribution before SMOTE:")
print(pd.Series(y_train).value_counts())

print("\nClass distribution after SMOTE:")
print(pd.Series(y_train_resampled).value_counts())

Train and evaluate on the resampled data.

In [ ]:
result_resampled = train_and_evaluate_models(X_train_resampled, y_train_resampled, X_test_scaled, y_test)

Add this final comparison to the running chart.

In [ ]:
#@title Model Accuracy Comparison

models = list(result_ohe.keys()) # Use keys from ohe as it likely has all models
accuracies_dropped = [result_encoded_dropped.get(model, 0) for model in models]
accuracies_imputed = [result_imputed.get(model, 0) for model in models]
accuracies_imputed_scaled = [result_scaled.get(model, 0) for model in models]
accuracies_ohe = list(result_ohe.values())
accuracies_selected = [result_selected_features.get(model, 0) for model in models]
accuracies_resampled = [result_resampled.get(model, 0) for model in models]

x = np.arange(len(models))  # the label locations
width = 0.12  # the width of the bars

fig, ax = plt.subplots(figsize=(22, 10))

rects1 = ax.bar(x - width*2.5, accuracies_dropped, width, label='Dropped Missing Values')
rects2 = ax.bar(x - width*1.5, accuracies_imputed, width, label='Imputed Missing Values')
rects3 = ax.bar(x - width*0.5, accuracies_imputed_scaled, width, label='Imputed and Scaled Values')
rects5 = ax.bar(x + width*0.5, accuracies_ohe, width, label='One-Hot Encoded, Imputed and Scaled Values')
rects6 = ax.bar(x + width*1.5, accuracies_selected, width, label='Selected Features (OHE, Imputed, Scaled)')
rects7 = ax.bar(x + width*2.5, accuracies_resampled, width, label='SMOTE Oversampled, Imputed and Scaled Values')

# Add some text for labels, title and custom x-axis tick labels, etc.
ax.set_ylabel('Accuracy')
ax.set_title('Model Accuracy Comparison: Different Preprocessing Methods and Feature Selection')
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=45, ha="right")
ax.legend()

plt.tight_layout()
plt.show()

SMOTE oversampling helped in some cases, not so much in others. That's because the data is too imbalanced to begin with — creating synthetic values isn't a cure-all. For the next section, we will skip resampling, since for most models performance after resampling doesn't improve on this dataset.

## 4. Hyperparameter Tuning

Hyperparameter tuning is the process of finding the best "settings" for a machine learning model — like the number of trees, tree depth, or learning rate — which aren't learned from the data but chosen beforehand. The right tuning can drastically improve performance, while bad choices can hurt even a strong model. Common approaches include manual tweaking, grid search, random search, or smarter methods like Bayesian optimization. Tools like `GridSearchCV` or Optuna make the process easier and more systematic.

For our final test, we will tune the hyperparameters of one of our models — Random Forest — using grid search.

Realistically, hyperparameter tuning should be performed using a validation set (not the test set). We use 5-fold stratified cross-validation for this, so the test set stays completely unseen until the very end.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [None, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'criterion': ['gini', 'entropy']
}

rf_clf = RandomForestClassifier(random_state=42)

grid_search_rf = GridSearchCV(
    estimator=rf_clf,
    param_grid=param_grid_rf,
    cv=StratifiedKFold(5),
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search_rf.fit(X_train_resampled, y_train_resampled)

print("Best parameters found: ", grid_search_rf.best_params_)
print("Best cross-validation accuracy: ", grid_search_rf.best_score_)

best_rf_clf = grid_search_rf.best_estimator_
y_pred_tuned_rf = best_rf_clf.predict(X_test_scaled)
test_accuracy_tuned_rf = accuracy_score(y_test, y_pred_tuned_rf)
print(f"Test set accuracy: {test_accuracy_tuned_rf:.4f}")

Finally, see how the tuned Random Forest compares to every prior version of it across this whole notebook.

In [ ]:
#@title Random Forest Accuracy Comparison

rf_accuracies = {
    'Dropped Missing Values': result_encoded_dropped.get('RandomForestClassifier', 0),
    'Imputed Missing Values': result_imputed.get('RandomForestClassifier', 0),
    'Imputed and Scaled Values': result_scaled.get('RandomForestClassifier', 0),
    'One-Hot Encoded, Imputed and Scaled Values': result_ohe.get('RandomForestClassifier', 0),
    'Selected Features (OHE, Imputed, Scaled)': result_selected_features.get('RandomForestClassifier', 0),
    'SMOTE Oversampled, Imputed and Scaled Values': result_resampled.get('RandomForestClassifier', 0),
    'Tuned (SMOTE, Scaled)': test_accuracy_tuned_rf
}

labels = list(rf_accuracies.keys())
accuracies = list(rf_accuracies.values())

# Tableau 10 colors
colors = [
    "#1f77b4",  "#ff7f0e",  "#2ca02c",  "#d62728",
    "#9467bd",  "#8c564b",  "#e377c2"
]

plt.figure(figsize=(14, 7))
bars = plt.bar(labels, accuracies, color=colors)
plt.ylabel('Accuracy')
plt.title('Random Forest Accuracy Comparison Across Preprocessing Steps')
plt.xticks(rotation=45, ha="right")
plt.ylim(0.75, 0.9)  # Zoom in on the relevant range for better visibility

# Annotate bars with percentages
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height + 0.002,
        f"{height:.2%}",  # format as percentage
        ha='center', va='bottom', fontsize=10
    )

plt.tight_layout()
plt.show()

## 5. Practice Problems

1. Repeat section 2.3's imputation, but use `strategy='median'` instead of `strategy='mean'` for the numerical columns. Re-run `train_and_evaluate_models` — does it change which model performs best?
2. In section 2.6, `RFECV` selected a subset of features using logistic regression as the estimator. Try using `RandomForestClassifier` as the RFECV estimator instead — does it select a different number or set of features?
3. In section 3, we only oversampled class 1 (`CL`) up to 80 samples. Try fully balancing all three classes to the same count (e.g. all at 188) and compare the resulting model accuracies to what we got with the partial oversampling used above.
4. Extend the `param_grid_rf` grid search in section 4 with one more hyperparameter of your choice (check the `RandomForestClassifier` docs for options like `max_features`). Does the best cross-validation accuracy improve?
5. Try `GridSearchCV` on a different model, such as `SVC` or `GradientBoostingClassifier`, using the resampled data from section 3. How does its tuned test accuracy compare to the tuned Random Forest above?